# 05_calidad
Validaciones de calidad de datos sobre la capa gold. Solo lee tablas: no escribe nada. Junta todas las fallas y recién al final corta con un error (`AssertionError`), para verlas todas juntas en una sola corrida.

Corre después de `04_gold`. Usa `ciclo` solo para decidir si le toca correr (ver "Control del ciclo"); no modifica el estado del ciclo.

## Tablas que lee

`gold_jugador`, `gold_debut`, `gold_aporte`, `gold_formacion_debut` y `gold_formacion_fifa`. También consulta `bronze_ciclos` (a través de `ciclo`).

## Contenido

| Bloque | Qué verifica |
|---|---|
| Control del ciclo | Que gold haya publicado hace poco; si no, el notebook se saltea |
| 1. Unicidad | Que haya una fila por clave en cada tabla (el grano de cada una) |
| 2. Fechas plausibles | Nacimiento, fallecimiento y temporada de debut dentro de rangos razonables |
| 3. Aportes | Claves completas y pesos entre 0 y 1, con suma por jugador que no supera 1 |
| Resultado final | Lanza un error con la lista de los chequeos que fallaron, o confirma que pasaron todos |

Cada chequeo imprime `OK` o `FALLA` con la cantidad de filas que violan la regla, y en los casos de falla muestra hasta 5 filas de ejemplo.

## Control del ciclo
`%run ./ciclo` carga las funciones del notebook `ciclo` y su widget `FORZAR`.

Las tablas gold solo cambian cuando `04_gold` publica un ciclo (`publicado_en`). Por eso este notebook valida solo si el ciclo vigente se publicó hace menos de `VENTANA_HORAS` horas (12 por defecto). Revalidar las mismas tablas todos los días daría el mismo resultado.

| Situación | Resultado |
|---|---|
| No hay ciclo, o el ciclo vigente todavía no se publicó | Termina con `SALTEADO: el ciclo vigente todavia no se publico` |
| El ciclo se publicó hace más de `VENTANA_HORAS` horas | Termina con `SALTEADO: ciclo <lunes> publicado en <fecha>, hace mas de <horas> h` |
| El ciclo se publicó dentro de la ventana | Imprime el ciclo y sigue con los chequeos |
| `FORZAR` vale `1` | Se saltea la compuerta y siempre valida |

Si un chequeo falla, la tarea termina en error ese día. En la corrida siguiente el ciclo ya está fuera de la ventana y el notebook se saltea: la falla queda en el historial del Job y no se repite sola.

In [ ]:
%run ./ciclo

In [ ]:
# Control del ciclo: solo valida cuando gold acaba de publicar.
from datetime import datetime, timedelta

dbutils.widgets.text("VENTANA_HORAS", "12")   # horas desde la publicacion durante las que se valida
VENTANA_HORAS = float(dbutils.widgets.get("VENTANA_HORAS"))

ciclo_v = ciclo_vigente()
if not forzar():
    if ciclo_v is None or not ciclo_v["publicado_en"]:
        dbutils.notebook.exit("SALTEADO: el ciclo vigente todavia no se publico")
    desde = (datetime.now() - timedelta(hours=VENTANA_HORAS)).isoformat(timespec="seconds")
    if ciclo_v["publicado_en"] < desde:
        dbutils.notebook.exit(f"SALTEADO: ciclo {ciclo_v['lunes']} publicado en {ciclo_v['publicado_en']}, hace mas de {VENTANA_HORAS:g} h")
print(f"ciclo {ciclo_v['lunes']} | publicado {ciclo_v['publicado_en']}" if ciclo_v else "sin ciclo (FORZAR)")

## Funciones y parámetros

| Elemento | Qué hace |
|---|---|
| `SCHEMA` | `workspace.inferiores` |
| `TOL` | `0.001`: tolerancia por el redondeo de las columnas `DECIMAL` |
| `fallas` | Lista donde se juntan los nombres de los chequeos que fallan |
| `t(nombre)` | Lee la tabla `workspace.inferiores.<nombre>` |
| `chequear(nombre, malas)` | Recibe un DataFrame con las filas que violan la regla. Si está vacío imprime `OK`; si no, imprime `FALLA`, la cantidad de filas, muestra hasta 5 y agrega el nombre a `fallas` |
| `duplicados(df, claves)` | Devuelve los grupos de claves que tienen más de una fila |

In [ ]:
# 05_calidad: validaciones de calidad de datos sobre la capa gold.
# Solo lee tablas, no escribe nada. Junta todas las fallas y corta al final
# con un error, para ver todo junto en una sola corrida.

from pyspark.sql import functions as F

SCHEMA = "workspace.inferiores"
TOL = 0.001  # tolerancia por redondeo de los DECIMAL
fallas = []


def t(nombre):
    return spark.table(f"{SCHEMA}.{nombre}")


def chequear(nombre, malas):
    """malas = DataFrame con las filas que violan la regla."""
    n = malas.count()
    if n == 0:
        print(f"OK     {nombre}")
    else:
        print(f"FALLA  {nombre}: {n} filas")
        malas.show(5, truncate=False)
        fallas.append(nombre)


def duplicados(df, claves):
    return df.groupBy(*claves).count().filter("count > 1")

## 1. Unicidad
Cada tabla tiene que tener una sola fila por clave.

| Tabla | Clave |
|---|---|
| `gold_jugador` | `player_id` |
| `gold_debut` | `player_id` |
| `gold_aporte` | `player_id`, `club_familia_id` |
| `gold_formacion_debut` | `player_id`, `anio`, `club_familia_id` |
| `gold_formacion_fifa` | `player_id`, `anio`, `club_familia_id` |

In [ ]:
# ---------------------------------------------------------------------------
# 1. Unicidad: una fila por clave (el grano de cada tabla)
# ---------------------------------------------------------------------------
chequear("gold_jugador: player_id único", duplicados(t("gold_jugador"), ["player_id"]))
chequear("gold_debut: player_id único", duplicados(t("gold_debut"), ["player_id"]))
chequear(
    "gold_aporte: (player_id, club_familia_id) único",
    duplicados(t("gold_aporte"), ["player_id", "club_familia_id"]),
)
chequear(
    "gold_formacion_debut: (player_id, anio, club_familia_id) único",
    duplicados(t("gold_formacion_debut"), ["player_id", "anio", "club_familia_id"]),
)
chequear(
    "gold_formacion_fifa: (player_id, anio, club_familia_id) único",
    duplicados(t("gold_formacion_fifa"), ["player_id", "anio", "club_familia_id"]),
)

## 2. Fechas plausibles
Las filas con la fecha nula no se consideran.

| Tabla | Regla |
|---|---|
| `gold_jugador` | `date_of_birth` entre el 1 de enero de 1950 y hoy |
| `gold_jugador` | `date_of_death` no es anterior a `date_of_birth` (solo si las dos fechas existen) |
| `gold_debut` | `temporada_debut` entre 1985 y el año próximo (el año actual más 1) |

In [ ]:
# ---------------------------------------------------------------------------
# 2. Fechas plausibles
# ---------------------------------------------------------------------------
jug = t("gold_jugador")
chequear(
    "gold_jugador: date_of_birth entre 1950 y hoy",
    jug.filter(
        F.col("date_of_birth").isNotNull()
        & ((F.col("date_of_birth") < F.lit("1950-01-01").cast("date"))
           | (F.col("date_of_birth") > F.current_date()))
    ),
)
chequear(
    "gold_jugador: date_of_death no es anterior a date_of_birth",
    jug.filter(
        F.col("date_of_death").isNotNull()
        & F.col("date_of_birth").isNotNull()
        & (F.col("date_of_death") < F.col("date_of_birth"))
    ),
)
chequear(
    "gold_debut: temporada_debut entre 1985 y el año próximo",
    t("gold_debut").filter(
        F.col("temporada_debut").isNotNull()
        & ((F.col("temporada_debut") < 1985)
           | (F.col("temporada_debut") > F.year(F.current_date()) + 1))
    ),
)

## 3. Aportes: claves completas y pesos coherentes
Sobre `gold_aporte`. Los cuatro pesos son `fifa` y `fifa_sin_relleno` (de `gold_formacion_fifa`) y `debut` y `debut_sin_relleno` (de `gold_formacion_debut`).

| Regla | Detalle |
|---|---|
| Claves completas | `player_id` y `club_familia_id` no son nulos |
| Peso entre 0 y 1 | Cada peso es la parte que se lleva un club de un jugador. Se acepta hasta `1 + TOL` |
| Suma por jugador | La suma de cada peso por jugador no supera `1 + TOL` |

No se exige que la suma sea 1: con datos incompletos o en las variantes `sin_relleno` puede ser menor, y eso es legítimo.

In [ ]:
# ---------------------------------------------------------------------------
# 3. Aportes: claves completas y pesos coherentes
# ---------------------------------------------------------------------------
ap = t("gold_aporte")
PESOS = ["fifa", "fifa_sin_relleno", "debut", "debut_sin_relleno"]

chequear(
    "gold_aporte: player_id y club_familia_id no nulos",
    ap.filter(F.col("player_id").isNull() | F.col("club_familia_id").isNull()),
)

# Cada peso es la parte que se lleva un club de un jugador: va de 0 a 1.
for c in PESOS:
    chequear(
        f"gold_aporte: {c} entre 0 y 1",
        ap.filter((F.col(c) < 0) | (F.col(c) > 1 + TOL)),
    )

# La suma por jugador no puede superar 1. No exigimos que sea 1: con datos
# incompletos o en las variantes sin_relleno puede ser menor, y eso es legítimo.
sumas = ap.groupBy("player_id").agg(*[F.sum(c).alias(c) for c in PESOS])
for c in PESOS:
    chequear(
        f"gold_aporte: suma de {c} por jugador no supera 1",
        sumas.filter(F.col(c) > 1 + TOL),
    )

## Resultado final
Si `fallas` tiene elementos, lanza un `AssertionError` con la cantidad de chequeos que fallaron y sus nombres, así la tarea del Job termina en error. Si no, imprime `Todos los chequeos de calidad pasaron.`

In [ ]:
# ---------------------------------------------------------------------------
# Resultado final
# ---------------------------------------------------------------------------
if fallas:
    raise AssertionError(
        f"Fallaron {len(fallas)} chequeos de calidad: " + " | ".join(fallas)
    )
print("Todos los chequeos de calidad pasaron.")